# 07 — Time series II: exponential smoothing, damped trend, Kalman (1950s–1985)

## Usage and numerical contracts
Charts migrated to the PlotSharp source build: NaN Y breaks the line without interpolating the gap; Infinity and entirely missing series are rejected. Release build of MathemaSharp/PlotSharp, then dotnet run --file ops/RunMigratedCoreNotebook.cs -- --smoothing. Six SVGs and an HTML table verified; earlier preview packages do not include this extension.
Smoothing and Kalman work in IEEE T, but the choice of parameters remains part of the model. SES/Holt/Holt-Winters have initialisation conventions: do not confuse theoretical identities with equivalence for any initialisation. Holt-Winters here is additive. In Kalman, missing observations skip the update; covariances and dimensions must be consistent. The filter produces causal estimates, it is not an RTS smoother. The didactic optimisation of the parameters does not guarantee a global maximum.
[API catalogue](12-api-reference.ipynb), [checks on the six formats](11-consolidation.ipynb), [limits and coverage](../../ARCHITECTURE.md).
**JupyterSharp** kernel (C#) on the **.NET 11** runtime. Prerequisite: `dotnet build src/MathemaSharp.slnx -c Release`. From a terminal: `scripts/run-notebooks.ps1` (sets the roll-forward to runtime 11); in VS Code you need `DOTNET_ROLL_FORWARD=LatestMajor` and, as long as .NET 11 is RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

First the **level** is updated, then **trend** (Holt, 1957) and **seasonality** (Winters, 1960) are added. In 1960 the **Kalman filter** formalises "predict and correct" on a hidden state. The two families turn out to be the same thing seen from two sides.

In [ ]:
#load "../shared/MathemaSharp.PlotSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
using MathemaSharp.TimeSeries;
// Local level: a level that walks at random (variance q) observed with noise (variance r).
const double q = 0.5, r = 2.0;
var rng = new Random(1960);
var level = 0.0;
var truth = new double[200];
var obs = new double[200];
for (var t = 0; t < 200; t++)
{
    level += Math.Sqrt(q) * Simulation.StandardNormal(rng);
    truth[t] = level;
    obs[t] = level + Math.Sqrt(r) * Simulation.StandardNormal(rng);
}
Chart.Lines("Local level: hidden level and observations", new Line("observations", obs), new Line("true level", truth));

Local level: hidden level and observations 0 5 10 15 20 0 50 100 150 200 <path d='M64 188.48L67.16 205.55L70.32 203.89L73.48 179.53L76.64 194.67L79.8 197.08L82.96 189.6L86.12 204.57L89.28 199.22L92.44 172.87L95.6 182.38L98.76 187.76L101.92 187.71L105.08 177.14L108.24 179.28L111.4 166.19L114.56 177.42L117.72 189.95L120.88 177.12L124.04 168.09L127.2 171.77L130.36 174.63L133.52 164.05L136.68 158.33L139.84 166.11L143 166.92L146.16 164.64L149.32 196.53L152.48 197.57L155.64 182.19L158.8 173.34L161.96 171.49L165.12 190.19L168.28 179.06L171.44 148.41L174.6 167.06L177.76 167.38L180.92 181.77L184.08 164.69L187.24 167.62L190.4 155.98L193.56 168.55L196.72 180.57L199.88 188.72L203.04 188.77L206.2 175.19L209.36 168.71L212.52 152.65L215.68 156.86L218.84 138.5L222 135.94L225.16 144.47L228.32 159.68L231.48 142.05L234.64 165.31L237.8 171.48L240.96 163.5L244.12 176.19L247.28 157.16L250.44 154.94L253.6 174.76L256.76 175.6L259.92 156.2L263.08 161.86L266.24 149.6L269.4 164.42L272.56 164.67L275.72 161.69L278.88 155.91L282.04 163.55L285.2 142.83L288.36 140.48L291.52 154.41L294.68 164.76L297.84 129.43L301 137.3L304.16 127.48L307.32 148.37L310.48 136.94L313.64 128.45L316.8 137.44L319.96 147.98L323.12 156.74L326.28 146.69L329.44 143.95L332.6 133.13L335.76 130.44L338.92 139.76L342.08 134.84L345.24 141.33L348.4 131.87L351.56 115.18L354.72 129.44L357.88 114.87L361.04 133.44L364.2 115.55L367.36 97.34L370.52 101.13L373.68 117.24L376.84 103.16L380 110.62L383.16 120.91L386.32 129.06L389.48 110L392.64 97.81L395.8 114.01L398.96 131.17L402.12 119.9L405.28 121.98L408.44 116.59L411.6 119.87L414.76 112.37L417.92 115.48L421.08 113.22L424.24 120.98L427.4 102.3L430.56 123.7L433.72 115.67L436.88 134.28L440.04 132.19L443.2 152.52L446.36 129.98L449.52 150.08L452.68 152.71L455.84 131.91L459 126.99L462.16 141.45L465.32 138.28L468.48 137.35L471.64 140.29L474.8 152.94L477.96 137.56L481.12 148.19L484.28 137.35L487.44 120.04L490.6 108.27L493.76 116.66L496.92 122.31L500.08 131.98L503.24 136.21L506.4 122.84L509.56 92.96L512.72 107.03L515.88 129.49L519.04 106.73L522.2 138.86L525.36 139.83L528.52 156.06L531.68 142.2L534.84 134.77L538 141.9L541.16 119.05L544.32 144.96L547.48 165.05L550.64 161.4L553.8 145.33L556.96 147.48L560.12 165.7L563.28 159.18L566.44 158.1L569.6 142.95L572.76 133.24L575.92 166.51L579.08 183.98L582.24 131.07L585.4 130.43L588.56 156.11L591.72 147.39L594.88 148.88L598.04 159.82L601.2 150.28L604.36 149.59L607.52 172.11L610.68 154.62L613.84 158.06L617 158.01L620.16 163.25L623.32 159.38L626.48 149.5L629.64 165.88L632.8 135.91L635.96 128.52L639.12 127.58L642.28 131.82L645.44 146.36L648.6 135.64L651.76 131.21L654.92 152.42L658.08 166.75L661.24 141.37L664.4 147.33L667.56 138.42L670.72 140.02L673.88 144.81L677.04 131.09L680.2 117.75L683.36 123.1L686.52 134.5L689.68 125.51L692.84 118.91' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 203.32L67.16 205.37L70.32 199.77L73.48 199.1L76.64 196.11L79.8 197.1L82.96 192.52L86.12 202.44L89.28 198.49L92.44 193.46L95.6 193.48L98.76 191.02L101.92 192.06L105.08 191.57L108.24 184.1L111.4 179.63L114.56 183.28L117.72 182.79L120.88 180.27L124.04 178.67L127.2 164.86L130.36 167.16L133.52 161.74L136.68 161.33L139.84 164.5L143 165.02L146.16 173.72L149.32 178.56L152.48 175.48L155.64 172.59L158.8 182.89L161.96 179.84L165.12 177.38L168.28 178.13L171.44 169.1L174.6 178.04L177.76 182.06L180.92 180.42L184.08 180.61L187.24 171.46L190.4 172L193.56 172.12L196.72 170.59L199.88 174.34L203.04 180.09L206.2 175.3L209.36 176.03L212.52 172.76L215.68 162.59L218.84 155.12L222 152.08L225.16 155.27L228.32 160.79L231.48 163.53L234.64 170.24L237.8 158.77L240.96 162.56L244.12 163.09L247.28 160.33L250.44 157.85L253.6 162.35L256.76 166.48L259.92 164.76L263.08 161.46L266.24 162.12L269.4 164.6L272.56 158.9L275.72 155.82L278.88 144.57L282.04 150.13L285.2 146.69L288.36 149.32L291.52 141.25L294.68 144.92L297.84 138.07L301 142.02L304.16 129.67L307.32 135.12L310.48 132.03L313.64 138.2L316.8 143.83L319.96 148.49L323.1

## SES: an average with geometric weights
$$\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1} \;\Longrightarrow\; \ell_t = \sum_{j\ge0}\alpha(1-\alpha)^j y_{t-j}, \qquad \hat y_{T+h}=\ell_T.$$
$\alpha$ is estimated by minimising the SSE of the one-step errors. `FitSesAlpha` uses the **golden section**: each iteration shrinks the interval by the factor $1/\varphi\approx0.618$, reusing a point already evaluated. It presupposes a unimodal SSE in $\alpha$.

In [ ]:
var grid = Enumerable.Range(1, 99).Select(i => i / 100.0).ToArray();
Chart.Lines("One-step SSE as a function of α (x axis: α × 100)", new Line("SSE", grid.Select(a => ExponentialSmoothing.Ses(obs, a, 0).Sse).ToArray(), Start: 1));
var alphaHat = ExponentialSmoothing.FitSesAlpha(obs);
var kStar = KalmanFilter.LocalLevelSteadyStateGain(q, r);
Console.WriteLine($"α estimated (golden section) = {alphaHat:F4}");
Console.WriteLine($"K* theoretical (Riccati, q/r = {q / r}) = {kStar:F4}");

One-step SSE as a function of α (x axis: α × 100) 500 1,000 1,500 2,000 2,500 3,000 3,500 0 20 40 60 80 100 <path d='M70.32 72.31L76.64 134.38L82.96 157.92L89.28 170.15L95.6 177.56L101.92 182.49L108.24 186L114.56 188.61L120.88 190.63L127.2 192.25L133.52 193.57L139.84 194.66L146.16 195.59L152.48 196.38L158.8 197.07L165.12 197.66L171.44 198.19L177.76 198.65L184.08 199.06L190.4 199.43L196.72 199.76L203.04 200.06L209.36 200.32L215.68 200.56L222 200.77L228.32 200.97L234.64 201.14L240.96 201.29L247.28 201.43L253.6 201.56L259.92 201.67L266.24 201.77L272.56 201.86L278.88 201.93L285.2 202L291.52 202.06L297.84 202.1L304.16 202.14L310.48 202.17L316.8 202.2L323.12 202.21L329.44 202.22L335.76 202.23L342.08 202.22L348.4 202.21L354.72 202.2L361.04 202.18L367.36 202.16L373.68 202.13L380 202.09L386.32 202.05L392.64 202.01L398.96 201.96L405.28 201.91L411.6 201.85L417.92 201.79L424.24 201.73L430.56 201.66L436.88 201.59L443.2 201.51L449.52 201.43L455.84 201.35L462.16 201.27L468.48 201.18L474.8 201.09L481.12 200.99L487.44 200.89L493.76 200.79L500.08 200.69L506.4 200.58L512.72 200.47L519.04 200.36L525.36 200.25L531.68 200.13L538 200.01L544.32 199.88L550.64 199.76L556.96 199.63L563.28 199.49L569.6 199.36L575.92 199.22L582.24 199.08L588.56 198.94L594.88 198.79L601.2 198.64L607.52 198.49L613.84 198.33L620.16 198.17L626.48 198.01L632.8 197.84L639.12 197.67L645.44 197.5L651.76 197.32L658.08 197.14L664.4 196.96L670.72 196.77L677.04 196.58L683.36 196.39L689.68 196.19' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> SSE · x=1: 3,050 SSE · x=2: 1,900 SSE · x=3: 1,464 SSE · x=4: 1,238 SSE · x=5: 1,101 SSE · x=6: 1,009 SSE · x=7: 944 SSE · x=8: 896 SSE · x=9: 859 SSE · x=10: 829 SSE · x=11: 804 SSE · x=12: 784 SSE · x=13: 767 SSE · x=14: 752 SSE · x=15: 740 SSE · x=16: 728 SSE · x=17: 719 SSE · x=18: 710 SSE · x=19: 703 SSE · x=20: 696 SSE · x=21: 690 SSE · x=22: 684 SSE · x=23: 679 SSE · x=24: 675 SSE · x=25: 671 SSE · x=26: 667 SSE · x=27: 664 SSE · x=28: 661 SSE · x=29: 659 SSE · x=30: 656 SSE · x=31: 654 SSE · x=32: 652 SSE · x=33: 651 SSE · x=34: 649 SSE · x=35: 648 SSE · x=36: 647 SSE · x=37: 646 SSE · x=38: 646 SSE · x=39: 645 SSE · x=40: 644 SSE · x=41: 644 SSE · x=42: 644 SSE · x=43: 644 SSE · x=44: 644 SSE · x=45: 644 SSE · x=46: 644 SSE · x=47: 645 SSE · x=48: 645 SSE · x=49: 646 SSE · x=50: 646 SSE · x=51: 647 SSE · x=52: 648 SSE · x=53: 649 SSE · x=54: 650 SSE · x=55: 651 SSE · x=56: 652 SSE · x=57: 653 SSE · x=58: 654 SSE · x=59: 656 SSE · x=60: 657 SSE · x=61: 659 SSE · x=62: 660 SSE · x=63: 662 SSE · x=64: 663 SSE · x=65: 665 SSE · x=66: 667 SSE · x=67: 669 SSE · x=68: 670 SSE · x=69: 672 SSE · x=70: 674 SSE · x=71: 676 SSE · x=72: 678 SSE · x=73: 681 SSE · x=74: 683 SSE · x=75: 685 SSE · x=76: 687 SSE · x=77: 690 SSE · x=78: 692 SSE · x=79: 695 SSE · x=80: 697 SSE · x=81: 700 SSE · x=82: 702 SSE · x=83: 705 SSE · x=84: 708 SSE · x=85: 710 SSE · x=86: 713 SSE · x=87: 716 SSE · x=88: 719 SSE · x=89: 722 SSE · x=90: 725 SSE · x=91: 728 SSE · x=92: 731 SSE · x=93: 735 SSE · x=94: 738 SSE · x=95: 741 SSE · x=96: 745 SSE · x=97: 748 SSE · x=98: 752 SSE · x=99: 756

α estimated (golden section) = 0.4301
K* theoretical (Riccati, q/r = 0.25) = 0.3904

## Kalman: why $\hat\alpha \approx K^*$
The local level is $\ell_t=\ell_{t-1}+\eta_t$, $y_t=\ell_t+\varepsilon_t$. The filter alternates:
- **predict**: $P^- = P + q$;
- **update**: $K = P^-/(P^-+r)$, $\ell \leftarrow \ell + K(y-\ell)$, $P = (1-K)P^-$.

The level update is *exactly* one line of SES with $\alpha=K$. In steady state $P^-$ satisfies the Riccati equation $P = P - P^2/(P+r) + q$, that is $P^2-qP-qr=0$:
$$P^* = \frac{q+\sqrt{q^2+4qr}}{2}, \qquad K^* = \frac{P^*}{P^*+r}.$$
So **SES = the stationary Kalman filter of the local level**, and $\alpha$ depends only on the signal-to-noise ratio $q/r$. Estimating $\alpha$ with the SSE is equivalent to estimating $q/r$. The library uses the covariance in **Joseph** form, $(I-KH)P(I-KH)^\top + KRK^\top$: algebraically equal, but it stays symmetric and positive semidefinite even with rounding.

In [ ]:
var kf = KalmanFilter.LocalLevel(q, r, obs[0], 1e6);
var filtered = kf.Filter(obs);
Chart.Lines("Gain K_t: from the diffuse prior (K≈1) to the steady state K*", new Line("K_t", filtered.FirstGain[..30]), new Line("K*", Enumerable.Repeat(kStar, 30).ToArray(), Dashed: true));
var ses = ExponentialSmoothing.Ses(obs, kStar, 0);
Console.WriteLine($"max |Kalman − SES(α=K*)| on the one-step forecasts, t ≥ 50: {Enumerable.Range(50, 150).Max(t => Math.Abs(filtered.Predictions[t] - ses.Fitted[t])):E2}");

Gain K_t: from the diffuse prior (K≈1) to the steady state K* 0.2 0.4 0.6 0.8 1 0 10 20 30 <path d='M64 175.73L85.07 175.73L106.13 175.73L127.2 175.73L148.27 175.73L169.33 175.73L190.4 175.73L211.47 175.73L232.53 175.73L253.6 175.73L274.67 175.73L295.73 175.73L316.8 175.73L337.87 175.73L358.93 175.73L380 175.73L401.07 175.73L422.13 175.73L443.2 175.73L464.27 175.73L485.33 175.73L506.4 175.73L527.47 175.73L548.53 175.73L569.6 175.73L590.67 175.73L611.73 175.73L632.8 175.73L653.87 175.73L674.93 175.73' fill='none' stroke='#eb6834' stroke-width='2' stroke-dasharray='6 4' stroke-linejoin='round' stroke-linecap='round'/> K_t · x=0: 1 K_t · x=1: 0.556 K_t · x=2: 0.446 K_t · x=3: 0.41 K_t · x=4: 0.398 K_t · x=5: 0.393 K_t · x=6: 0.391 K_t · x=7: 0.391 K_t · x=8: 0.391 K_t · x=9: 0.39 K_t · x=10: 0.39 K_t · x=11: 0.39 K_t · x=12: 0.39 K_t · x=13: 0.39 K_t · x=14: 0.39 K_t · x=15: 0.39 K_t · x=16: 0.39 K_t · x=17: 0.39 K_t · x=18: 0.39 K_t · x=19: 0.39 K_t · x=20: 0.39 K_t · x=21: 0.39 K_t · x=22: 0.39 K_t · x=23: 0.39 K_t · x=24: 0.39 K_t · x=25: 0.39 K_t · x=26: 0.39 K_t · x=27: 0.39 K_t · x=28: 0.39 K_t · x=29: 0.39 K* · x=0: 0.39 K* · x=1: 0.39 K* · x=2: 0.39 K* · x=3: 0.39 K* · x=4: 0.39 K* · x=5: 0.39 K* · x=6: 0.39 K* · x=7: 0.39 K* · x=8: 0.39 K* · x=9: 0.39 K* · x=10: 0.39 K* · x=11: 0.39 K* · x=12: 0.39 K* · x=13: 0.39 K* · x=14: 0.39 K* · x=15: 0.39 K* · x=16: 0.39 K* · x=17: 0.39 K* · x=18: 0.39 K* · x=19: 0.39 K* · x=20: 0.39 K* · x=21: 0.39 K* · x=22: 0.39 K* · x=23: 0.39 K* · x=24: 0.39 K* · x=25: 0.39 K* · x=26: 0.39 K* · x=27: 0.39 K* · x=28: 0.39 K* · x=29: 0.39 K* K_t

max |Kalman − SES(α=K*)| on the one-step forecasts, t ≥ 50: 1.57E-011

## Missing data: the advantage of the state-space model
With a missing observation the Kalman filter skips the update: the state stays at the prediction and the variance $S_t = P^-_t + r$ grows by $q$ at every empty step. "Recipe" exponential smoothing has no natural way to do this. The filter also returns the log-likelihood by prediction-error decomposition, $-\tfrac12\sum(\ln 2\pi S_t + v_t^2/S_t)$: this is what is maximised to estimate $q$ and $r$.

In [ ]:
var gappy = (double[])obs.Clone();
for (var t = 80; t < 110; t++) gappy[t] = double.NaN;
var fg = KalmanFilter.LocalLevel(q, r, obs[0], 1e6).Filter(gappy);
// From t = 5: with the diffuse prior (variance 1e6) the initial band is enormous and would crush the scale.
var upper = fg.Predictions.Select((p, t) => p + 2 * Math.Sqrt(fg.PredictionVariances[t])).ToArray();
var lower = fg.Predictions.Select((p, t) => p - 2 * Math.Sqrt(fg.PredictionVariances[t])).ToArray();
Chart.Lines("One-step forecast ± 2√S with 30 missing observations (t = 80…109)",
    new Line("observations", gappy[5..], Start: 5),
    new Line("forecast", fg.Predictions[5..], Start: 5),
    new Line("+2√S", upper[5..], Start: 5, Dashed: true),
    new Line("−2√S", lower[5..], Start: 5, Dashed: true));
Console.WriteLine($"log-likelihood: q=0.5 → {KalmanFilter.LocalLevel(0.5, 2, obs[0], 1e6).Filter(obs).LogLikelihood:F2};  q=0.05 → {KalmanFilter.LocalLevel(0.05, 2, obs[0], 1e6).Filter(obs).LogLikelihood:F2};  q=5 → {KalmanFilter.LocalLevel(5, 2, obs[0], 1e6).Filter(obs).LogLikelihood:F2}");

One-step forecast ± 2√S with 30 missing observations (t = 80…109) -5 0 5 10 15 20 0 50 100 150 200 <path d='M79.8 170.86L82.96 164.88L86.12 176.85L89.28 172.57L92.44 151.49L95.6 159.1L98.76 163.41L101.92 163.37L105.08 154.91L108.24 156.62L111.4 146.15L114.56 155.14L117.72 165.16L120.88 154.9L124.04 147.67L127.2 150.62L130.36 152.91L133.52 144.44L136.68 139.86L139.84 146.08L143 146.74L146.16 144.91L149.32 170.43L152.48 171.26L155.64 158.95L158.8 151.87L161.96 150.39L165.12 165.36L168.28 156.44L171.44 131.93L174.6 146.85L177.76 147.1L180.92 158.61L184.08 144.95L187.24 147.3L190.4 137.99L193.56 148.04L196.72 157.66L199.88 164.18L203.04 164.21L206.2 153.35L209.36 148.16L212.52 135.32L215.68 138.69L218.84 124L222 121.95L225.16 128.77L228.32 140.94L231.48 126.84L234.64 145.45L237.8 150.39L240.96 144L244.12 154.15L247.28 138.93L250.44 137.15L253.6 153.01L256.76 153.68L259.92 138.16L263.08 142.69L266.24 132.88L269.4 144.74L272.56 144.94L275.72 142.55L278.88 137.93L282.04 144.04L285.2 127.47L288.36 125.59L291.52 136.73L294.68 145.01L297.84 116.74L301 123.04L304.16 115.19L307.32 131.9L310.48 122.75L313.64 115.96M411.6 109.09L414.76 103.1L417.92 105.58L421.08 103.77L424.24 109.99L427.4 95.04L430.56 112.16L433.72 105.74L436.88 120.62L440.04 118.95L443.2 135.21L446.36 117.18L449.52 133.26L452.68 135.37L455.84 118.73L459 114.79L462.16 126.36L465.32 123.82L468.48 123.08L471.64 125.43L474.8 135.55L477.96 123.25L481.12 131.75L484.28 123.08L487.44 109.23L490.6 99.82L493.76 106.53L496.92 111.05L500.08 118.78L503.24 122.17L506.4 111.47L509.56 87.57L512.72 98.83L515.88 116.79L519.04 98.58L522.2 124.29L525.36 125.06L528.52 138.05L531.68 126.96L534.84 121.02L538 126.72L541.16 108.44L544.32 129.17L547.48 145.24L550.64 142.32L553.8 129.47L556.96 131.18L560.12 145.76L563.28 140.55L566.44 139.68L569.6 127.56L572.76 119.79L575.92 146.41L579.08 160.39L582.24 118.05L585.4 117.54L588.56 138.09L591.72 131.11L594.88 132.3L598.04 141.06L601.2 133.43L604.36 132.87L607.52 150.89L610.68 136.9L613.84 139.65L617 139.61L620.16 143.8L623.32 140.7L626.48 132.8L629.64 145.9L632.8 121.93L635.96 116.02L639.12 115.26L642.28 118.66L645.44 130.28L648.6 121.71L651.76 118.17L654.92 135.14L658.08 146.6L661.24 126.3L664.4 131.07L667.56 123.93L670.72 125.21L673.88 129.05L677.04 118.07L680.2 107.4L683.36 111.68L686.52 120.8L689.68 113.61L692.84 108.33' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M79.8 167.63L82.96 168.9L86.12 167.33L89.28 171.05L92.44 171.64L95.6 163.78L98.76 161.95L101.92 162.52L105.08 162.85L108.24 159.75L111.4 158.53L114.56 153.7L117.72 154.26L120.88 158.51L124.04 157.1L127.2 153.42L130.36 152.33L133.52 152.55L136.68 149.38L139.84 145.67L143 145.83L146.16 146.19L149.32 145.69L152.48 155.35L155.64 161.56L158.8 160.54L161.96 157.16L165.12 154.51L168.28 158.75L171.44 157.85L174.6 147.73L177.76 147.38L180.92 147.27L184.08 151.7L187.24 149.06L190.4 148.37L193.56 144.32L196.72 145.77L199.88 150.41L203.04 155.79L206.2 159.08L209.36 156.84L212.52 153.45L215.68 146.38L218.84 143.37L222 135.81L225.16 130.4L228.32 129.76L231.48 134.13L234.64 131.28L237.8 136.81L240.96 142.11L244.12 142.85L247.28 147.26L250.44 144.01L253.6 141.33L256.76 145.89L259.92 148.93L263.08 144.73L266.24 143.93L269.4 139.62L272.56 141.62L275.72 142.91L278.88 142.77L282.04 140.88L285.2 142.11L288.36 136.4L291.52 132.18L294.68 133.95L297.84 138.27L301 129.87L304.16 127.2L307.32 122.51L310.48 126.17L313.64 124.84L316.8 121.37L319.96 121.37L323.12 121.37L326.28 121.37L329.44 121.37L332.6 121.37L335.76 121.37L338.92 121.37L342.08 121.37L345.24 121.37L348.4 121.37L351.56 121.37L354.72 121.37L357.88 121.37L361.04 121.37L364.2 121.37L367.36 121.37L370.52 121.37L373.68 121.37L376.84 121.37L380 121.37L383.16 121.37L386.32 121.37L389.48 121.37L392.64 121.37L395.8 121.37L398.96 121.37L402.12 121.37L405.28 121.37L408.44 121.37L411.6 121.37L414.76 110.44L417.92 106.53L421.08 106.11L424.24 105.16L427.4 107.08L430.56 102.35L433.72 106.19L436.88 106.01L440.

log-likelihood: q=0.5 → -407.24;  q=0.05 → -434.79;  q=5 → -444.37

## Holt and the damped trend (Gardner & McKenzie, 1985)
$$\hat y_{t|t-1}=\ell_{t-1}+\phi b_{t-1},\quad \ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+\phi b_{t-1}),\quad b_t=\beta(\ell_t-\ell_{t-1})+(1-\beta)\phi b_{t-1}$$
$$\hat y_{T+h}=\ell_T+(\phi+\phi^2+\dots+\phi^h)\,b_T \;\xrightarrow{h\to\infty}\; \ell_T+\frac{\phi}{1-\phi}b_T.$$
With $\phi=1$ it is linear Holt: the trend continues to infinity. With $\phi<1$ the contribution of the trend shrinks at long horizons. Empirically (M-competitions) it is one of the most robust methods: rarely the best, almost never disastrous.

In [ ]:
// Slowing growth: the slope decays geometrically.
var slowing = Enumerable.Range(0, 60).Select(t => 50 + 40 * (1 - Math.Pow(0.96, t)) + Simulation.StandardNormal(rng)).ToArray();
double[] tr = slowing[..40], te = slowing[40..];
var holt = ExponentialSmoothing.Holt(tr, 0.4, 0.2, 20);
var damped = ExponentialSmoothing.Holt(tr, 0.4, 0.2, 20, phi: 0.85);
Chart.Lines("Linear Holt vs damped (φ = 0.85)", new Line("observed", slowing), new Line("Holt", holt.Forecast, Start: 40, Dashed: true), new Line("damped", damped.Forecast, Start: 40, Dashed: true));
Table.Show(new[] { "Method", "RMSE h=1..20", "limit h→∞" }, new[]
{
    new object[] { "Holt (φ=1)", Accuracy.Rmse(te, holt.Forecast), "∞" },
    new object[] { "Damped (φ=0.85)", Accuracy.Rmse(te, damped.Forecast), (damped.Level + 0.85 / 0.15 * damped.Trend).ToString("F2") },
});

Linear Holt vs damped (φ = 0.85) 50 60 70 80 90 100 0 20 40 60 <path d='M64 208.83L74.53 202.68L85.07 199.37L95.6 196.2L106.13 196.51L116.67 191.2L127.2 184.38L137.73 180.14L148.27 181.66L158.8 171.23L169.33 176.51L179.87 166.48L190.4 164.03L200.93 160.22L211.47 158.2L222 154.33L232.53 155.13L243.07 150.87L253.6 144.59L264.13 144.3L274.67 145.26L285.2 144.29L295.73 142.58L306.27 137.95L316.8 137.74L327.33 140.17L337.87 136.63L348.4 128.99L358.93 137.36L369.47 131.58L380 126.56L390.53 124.02L401.07 127.44L411.6 123.71L422.13 122.3L432.67 123.01L443.2 125.36L453.73 116.02L464.27 118.31L474.8 114.2L485.33 120.51L495.87 115.43L506.4 114.89L516.93 111.51L527.47 116.59L538 113.92L548.53 113.25L559.07 112.44L569.6 108.69L580.13 114.07L590.67 112.94L601.2 105.23L611.73 107.62L622.27 111.7L632.8 104.8L643.33 103.14L653.87 104.61L664.4 107.19L674.93 109.71L685.47 99.93' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> observed · x=0: 50.4 observed · x=1: 52.5 observed · x=2: 53.7 observed · x=3: 54.8 observed · x=4: 54.7 observed · x=5: 56.5 observed · x=6: 58.9 observed · x=7: 60.4 observed · x=8: 59.8 observed · x=9: 63.5 observed · x=10: 61.6 observed · x=11: 65.1 observed · x=12: 66 observed · x=13: 67.3 observed · x=14: 68 observed · x=15: 69.3 observed · x=16: 69.1 observed · x=17: 70.5 observed · x=18: 72.7 observed · x=19: 72.8 observed · x=20: 72.5 observed · x=21: 72.8 observed · x=22: 73.4 observed · x=23: 75 observed · x=24: 75.1 observed · x=25: 74.2 observed · x=26: 75.5 observed · x=27: 78.1 observed · x=28: 75.2 observed · x=29: 77.2 observed · x=30: 79 observed · x=31: 79.9 observed · x=32: 78.7 observed · x=33: 80 observed · x=34: 80.5 observed · x=35: 80.2 observed · x=36: 79.4 observed · x=37: 82.6 observed · x=38: 81.8 observed · x=39: 83.3 observed · x=40: 81.1 observed · x=41: 82.8 observed · x=42: 83 observed · x=43: 84.2 observed · x=44: 82.4 observed · x=45: 83.4 observed · x=46: 83.6 observed · x=47: 83.9 observed · x=48: 85.2 observed · x=49: 83.3 observed · x=50: 83.7 observed · x=51: 86.4 observed · x=52: 85.6 observed · x=53: 84.1 observed · x=54: 86.5 observed · x=55: 87.1 observed · x=56: 86.6 observed · x=57: 85.7 observed · x=58: 84.8 observed · x=59: 88.2 Holt · x=40: 83.3 Holt · x=41: 83.9 Holt · x=42: 84.4 Holt · x=43: 84.9 Holt · x=44: 85.5 Holt · x=45: 86 Holt · x=46: 86.6 Holt · x=47: 87.1 Holt · x=48: 87.7 Holt · x=49: 88.2 Holt · x=50: 88.7 Holt · x=51: 89.3 Holt · x=52: 89.8 Holt · x=53: 90.4 Holt · x=54: 90.9 Holt · x=55: 91.5 Holt · x=56: 92 Holt · x=57: 92.5 Holt · x=58: 93.1 Holt · x=59: 93.6 damped · x=40: 82.7 damped · x=41: 83 damped · x=42: 83.2 damped · x=43: 83.4 damped · x=44: 83.6 damped · x=45: 83.7 damped · x=46: 83.8 damped · x=47: 83.9 damped · x=48: 84 damped · x=49: 84.1 damped · x=50: 84.2 damped · x=51: 84.2 damped · x=52: 84.3 damped · x=53: 84.3 damped · x=54: 84.3 damped · x=55: 84.4 damped · x=56: 84.4 damped · x=57: 84.4 damped · x=58: 84.4 damped · x=59: 84.4 damped Holt observed

Method,RMSE h=1..20,limit h→∞
Holt (φ=1),4.3414,∞
Damped (φ=0.85),1.5348,84.51


## Additive Holt-Winters (1960)
It adds $s_t=\gamma(y_t-\ell_{t-1}-\phi b_{t-1})+(1-\gamma)s_{t-m}$ and $\hat y_{T+h}=\ell_T+\phi_h b_T+s_{T+h-m(k+1)}$. Initialisation: the mean of the first cycle estimates the level at the **centre** of the cycle, $(m-1)/2$, not at the end. Shifting it along the trend, a line + zero-mean seasonality is reproduced exactly: this is one of the library's tests.

In [ ]:
var seasonalY = Enumerable.Range(0, 96).Select(t => 20 + 0.3 * t + 5 * Math.Sin(2 * Math.PI * t / 12) + Simulation.StandardNormal(rng)).ToArray();
var hw = ExponentialSmoothing.HoltWinters(seasonalY[..84], 12, 0.3, 0.05, 0.2, 12);
Chart.Lines("Holt-Winters: one-step fitted and 12-month forecast", new Line("observed", seasonalY), new Line("fitted", hw.Fitted), new Line("forecast", hw.Forecast, Start: 84, Dashed: true));
Console.WriteLine($"MASE: Holt-Winters {Accuracy.Mase(seasonalY[84..], hw.Forecast, seasonalY[..84], 12):F3}, seasonal naive {Accuracy.Mase(seasonalY[84..], ReferenceForecasts.SeasonalNaive(seasonalY[..84], 12, 12), seasonalY[..84], 12):F3}");

Holt-Winters: one-step fitted and 12-month forecast 10 20 30 40 50 60 0 20 40 60 80 100 <path d='M64 186.61L70.32 175.78L76.64 165.07L82.96 167.32L89.28 165.48L95.6 174.21L101.92 174.32L108.24 178.69L114.56 183.31L120.88 186.94L127.2 183.92L133.52 179.62L139.84 169.56L146.16 164.97L152.48 154.14L158.8 151.32L165.12 155.86L171.44 156.51L177.76 166.04L184.08 171.53L190.4 174.53L196.72 180.55L203.04 174.91L209.36 166L215.68 157.3L222 151.05L228.32 144.47L234.64 139.25L240.96 142.36L247.28 152.5L253.6 155.57L259.92 159.55L266.24 166.01L272.56 167.19L278.88 167.72L285.2 155.29L291.52 148.93L297.84 138.85L304.16 138.37L310.48 136.36L316.8 136.58L323.12 145.98L329.44 149.81L335.76 149.7L342.08 156.95L348.4 152.65L354.72 152.61L361.04 143.32L367.36 144.33L373.68 130.43L380 121.1L386.32 122.01L392.64 117.98L398.96 131.81L405.28 134.47L411.6 141.81L417.92 144.99L424.24 145.76L430.56 142.91L436.88 139.81L443.2 128.52L449.52 120.07L455.84 111.93L462.16 110.41L468.48 112.7L474.8 114.92L481.12 124.92L487.44 129.57L493.76 134.7L500.08 134.15L506.4 125.24L512.72 127.08L519.04 116.7L525.36 113.92L531.68 103.41L538 98.38L544.32 101.3L550.64 108.61L556.96 119.41L563.28 116.55L569.6 126.52L575.92 122.62L582.24 122.8L588.56 114.36L594.88 106.81L601.2 104.31L607.52 96.32L613.84 91.59L620.16 93.25L626.48 100.14L632.8 103.06L639.12 109.55L645.44 116.7L651.76 116.89L658.08 109.95L664.4 104.64' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 186.61L70.32 175.78L76.64 165.07L82.96 167.32L89.28 165.48L95.6 174.21L101.92 174.32L108.24 178.69L114.56 183.31L120.88 186.94L127.2 183.92L133.52 179.62L139.84 175.33L146.16 162.68L152.48 152.6L158.8 155.29L165.12 152.17L171.44 161.97L177.76 160.32L184.08 166.38L190.4 172.6L196.72 176.88L203.04 175.1L209.36 170.87L215.68 164.02L222 152.74L228.32 141.3L234.64 143.38L240.96 141.75L247.28 148.75L253.6 152.2L259.92 157.5L266.24 162.15L272.56 167.41L278.88 163.67L285.2 159.82L291.52 152.76L297.84 143.45L304.16 132.16L310.48 133.79L316.8 135.32L323.12 143.36L329.44 146.58L335.76 151.76L342.08 155.67L348.4 159.43L354.72 154.57L361.04 147.1L367.36 140.32L373.68 133.24L380 124.7L386.32 122.57L392.64 122.76L398.96 129.03L405.28 132.2L411.6 135.8L417.92 142.68L424.24 145.03L430.56 143.39L436.88 136.04L443.2 133.22L449.52 122.18L455.84 113.72L462.16 112.79L468.48 111.62L474.8 121.27L481.12 121.57L487.44 126.23L493.76 131.52L500.08 133.78L506.4 131.75L512.72 123.31L519.04 118.67L525.36 108.87L531.68 102.64L538 102.42L544.32 101.47L550.64 109.27L556.96 113.32L563.28 118.94L569.6 122.53L575.92 124.54L582.24 120.48L588.56 116.9' fill='none' stroke='#eb6834' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> observed · x=0: 18.1 observed · x=1: 21.9 observed · x=2: 25.6 observed · x=3: 24.8 observed · x=4: 25.5 observed · x=5: 22.4 observed · x=6: 22.4 observed · x=7: 20.9 observed · x=8: 19.3 observed · x=9: 18 observed · x=10: 19.1 observed · x=11: 20.5 observed · x=12: 24 observed · x=13: 25.6 observed · x=14: 29.4 observed · x=15: 30.4 observed · x=16: 28.8 observed · x=17: 28.6 observed · x=18: 25.3 observed · x=19: 23.4 observed · x=20: 22.3 observed · x=21: 20.2 observed · x=22: 22.2 observed · x=23: 25.3 observed · x=24: 28.3 observed · x=25: 30.5 observed · x=26: 32.8 observed · x=27: 34.6 observed · x=28: 33.5 observed · x=29: 30 observed · x=30: 28.9 observed · x=31: 27.5 observed · x=32: 25.3 observed · x=33: 24.9 observed · x=34: 24.7 observed · x=35: 29 observed · x=36: 31.2 observed · x=37: 34.7 observed · x=38: 34.9 observed · x=39: 35.6 observed · x=40: 35.5 observed · x=41: 32.2 observed · x=42: 30.9 observed · x=43: 30.9 observed · x=44: 28.4 observed · x=45: 29.9 observed · x=46: 29.9 observed · x=47: 33.2 observed · x=48: 32.8 observed · x=49: 37.6 observed · x=50: 40.9 observed · x=51: 40.6 observed · x=52: 42 observed · x=53: 37.1 observed · x=54: 36.2 observed · x=55: 33.7 observed · x=56: 32.6 observed · x=57: 32.3 observed · x=58: 3

MASE: Holt-Winters 0.174, seasonal naive 0.883

## ETS and ARIMA: complementary families
SES is equivalent to ARIMA(0,1,1) with $\theta=\alpha-1$: from $\ell_t = y_t-(1-\alpha)e_t$ it follows that $\hat y_{t+1}=y_t+(\alpha-1)e_t$. Linear Holt is equivalent to ARIMA(0,2,2). The state-space ETS framework (Hyndman et al., 2002) unifies smoothing and likelihood. ETS and ARIMA are not each other's update: they overlap only on linear models.

In [ ]:
var sesF = ExponentialSmoothing.Ses(obs, 0.3, 5).Forecast;
var arimaF = new ArimaModel<double>(0, 1, 1, 0, Array.Empty<double>(), new[] { 0.3 - 1 }, 1).Forecast(obs, 5);
Console.WriteLine($"SES(α=0.3):            {string.Join(", ", sesF.Select(v => v.ToString("F10")))}");
Console.WriteLine($"ARIMA(0,1,1), θ=−0.7:  {string.Join(", ", arimaF.Select(v => v.ToString("F10")))}");

SES(α=0.3):            11.6039306841, 11.6039306841, 11.6039306841, 11.6039306841, 11.6039306841
ARIMA(0,1,1), θ=−0.7:  11.6039306841, 11.6039306841, 11.6039306841, 11.6039306841, 11.6039306841